# Generalized hourglass plots for MAF

- adapted from https://github.com/lsst/rubin_sim_notebooks/blob/main/maf/hourglass/hourglass_plots.ipynb
- creation : 2026-08-05
- last update : 2026-08-10
- author : Sylvie Dagoret-Campagne

## Setup of the notebook

### Imports

In [ ]:
import warnings
import logging
import urllib
from os import path
import numpy as np

from collections import OrderedDict
from tempfile import TemporaryDirectory
import matplotlib.pyplot as plt
from scipy.constants import golden

from rubin_sim import maf
import rubin_sim.maf.db
import rubin_sim.maf.metric_bundles
from rubin_sim.data import get_baseline

from rubin_sim.maf.metrics import BaseMetric

In [ ]:
%matplotlib inline
# %config InlineBackend.figure_format = 'svg'
# %load_ext lab_black
# %load_ext pycodestyle_magic
# %flake8_on --ignore E501,W505
%load_ext autoreload
%autoreload 1

### `jupyter` magic

### Logging

Include logging to get a better idea of how long things take. This is more accurate than jupyter `%%time` magic, because it `%%time` does not include the time it takes for the client to render the plot, but this time is part of what is experienced by the user.

In [ ]:
logging.basicConfig(format="%(asctime)s %(message)s")
logger = logging.getLogger("hourglass_notebook")
logger.setLevel("DEBUG")
logger.info("Starting")

### Plotting configuration

The limited default resolution of matplotlib images can distort the gaps between exposures. This can be fixed either by increasing the precision of PNG image, or using a vector based format like SVG.
Comment out whichever jupyter magic below corresponds to the method you don't want to use.
For plots with lots of data, SVGs can cause performance issues in some browsers.

If you want to use SVGs, the line in this cell should be uncommented out:

In [ ]:
# %config InlineBackend.figure_formats = ['svg']

If you want to use PNGs with high enough dpi to avoid pixel-level distortion, the lines in this cell can be uncommented out:

In [ ]:
# plt.rcParams['figure.dpi'] = 300
# plt.rcParams['savefig.dpi'] = 300

In [ ]:
plt.rcParams["backend"]

Note that increasing the pdi in a jupyter notebook will make the images very large when displayed within the notebook.

### Suppress useless warnings

In [ ]:
warnings.filterwarnings(
    "ignore",
    append=True,
    message=r".*Tried to get polar motions for times after IERS data is valid.*",
)
warnings.filterwarnings("ignore", append=True, message=r".*dubious year.*")

### Configuration

In [ ]:
YEAR = 2026
MONTH = 9

In [ ]:
data_dir = "."

If you have a local copy of the scheduler output you want to use, set `opsim_origin` to its path.

In [ ]:
maf_output_dir = path.join(data_dir, "maf_output")

- Download the data file if necessary:
- https://s3df.slac.stanford.edu/data/rubin/sim-data/rubin_sim_data/
- https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/maf/

## Get input and output database connections

In [ ]:
logger.debug("Configuring database connections")
results_db = maf.db.ResultsDb(out_dir=maf_output_dir)

baseline = get_baseline()

In [ ]:
# check which columns are available
import sqlite3

opsim_fname = baseline
conn = sqlite3.connect(opsim_fname)
cursor = conn.cursor()

cursor.execute("PRAGMA table_info(observations);")
cols = cursor.fetchall()

In [ ]:
list(cols)

In [ ]:
cols_to_check = ["scheduler_note", "target_name", "observation_reason", "science_program"]

for col in cols_to_check:
    try:
        print(f"\n=== {col} ===")
        # cursor.execute(f"SELECT DISTINCT {col} FROM observations;")

        cursor.execute(
            f"""
            SELECT {col}, COUNT(*) 
            FROM observations 
            GROUP BY {col}
            ORDER BY COUNT(*) DESC
            LIMIT 20;
            """
        )

        values = cursor.fetchall()
        for v in values:
            print(v[0])
    except Exception as e:
        print(f"{col} -> erreur ({e})")

### Median Seeing in 5 minute time intervals

Use the `TimeIntervaleSlicer` to divide the survey into 300 second (5 minute) time segments, and calculate the median seeing in each segment.
Make an hourglass plot of the result for October of 2023.

In [ ]:
logger.info("Starting hourglass")
bundle_group = maf.metric_bundles.MetricBundleGroup(
    bundle_dict=[
        maf.metric_bundles.MetricBundle(
            metric=maf.metrics.MedianMetric("seeingFwhmEff"),
            slicer=maf.slicers.TimeIntervalSlicer(interval_seconds=300),
            constraint="",
            plot_funcs=[maf.plots.MonthHourglassPlot(MONTH, YEAR)],
            plot_dict={"colorbar": True, "label": "seeingFwhmEff"},
        )
    ],
    db_con=baseline,
    out_dir=maf_output_dir,
    results_db=results_db,
)
bundle_group.run_all()
bundle_group.plot_all(closefigs=False)

In [ ]:
import numpy as np

np.array([]).__class__

In [ ]:
logger.info("Finished hourglass")

### Hour Angle by visit

Use the `VisitIntervaleSlicer` to divide the survey into time segments each of which covers one visit, and calculate the hour angle in each segment.
Make an hourglass plot of the result for October of 2023.

Set the color map to be diverging, and force the color limits such that 0 (transiting) is gray, while positive and negative hour angle are red and blue.

In [ ]:
logger.info("Starting hourglass")
bundle_group = maf.metric_bundles.MetricBundleGroup(
    bundle_dict=[
        maf.metric_bundles.MetricBundle(
            metric=maf.metrics.MedianMetric("HA"),
            slicer=maf.slicers.VisitIntervalSlicer(),
            constraint="",
            plot_dict={
                "cmap": plt.get_cmap("coolwarm"),
                "color_limits": (-4.5, 4.5),
                "colorbar": True,
                "label": "medianHA ",
            },
            plot_funcs=[maf.plots.MonthHourglassPlot(MONTH, YEAR)],
        )
    ],
    db_con=baseline,
    out_dir=maf_output_dir,
    results_db=results_db,
)
bundle_group.run_all()
bundle_group.plot_all(closefigs=False)

In [ ]:
logger.info("Finished hourglass")

### Filter metric

In [ ]:
# This could have been the Pass metric instead .. (just returns the values in a given set of columns)
class FilterUseMetric(maf.metrics.BaseMetric):
    """Metric to classify visits by filter"""

    def __init__(self, filter_col="band", **kwargs):
        self.filter_col = filter_col
        super().__init__(col=[filter_col], metric_dtype="object", **kwargs)

    def run(self, data_slice, slice_point=None):  # pylint: disable=invalid-name
        """Run the metric.

        Parameters
        ----------
        dataSlice : numpy.NDarray
           Values passed to metric by the slicer, which the metric will use to calculate
           metric values at each slicePoint.
        slicePoint : Dict
           Dictionary of slicePoint metadata passed to each metric.
           E.g. the ra/dec of the healpix pixel or opsim fieldId.

        Returns
        -------
        str
            use at each slicePoint.
        """
        return data_slice[self.filter_col]

In [ ]:
logger.info("Starting hourglass")
bundle_group = maf.metric_bundles.MetricBundleGroup(
    bundle_dict=[
        maf.metric_bundles.MetricBundle(
            metric=FilterUseMetric(),
            slicer=maf.slicers.VisitIntervalSlicer(),
            constraint="night < 1200",
            plot_dict={
                "cmap": plt.get_cmap("Set1"),
                "assigned_colors": OrderedDict([("u", 1), ("g", 2), ("r", 4), ("i", 7), ("z", 0), ("y", 3)]),
                "title": f"Filter usage for {MONTH}/{YEAR}",
                "legend_ncols": 1,
                "legend_loc": (1.01, 0.5),
                "legend_bbox_to_anchor": (1.01, 0.0),
                "legend": True,
            },
            plot_funcs=[maf.plots.MonthHourglassCategoricalPlot(MONTH, YEAR)],
        )
    ],
    db_con=baseline,
    out_dir=maf_output_dir,
    results_db=results_db,
)
bundle_group.run_all()
bundle_group.plot_all(closefigs=False)

In [ ]:
logger.info("Finish hourglass")

### Time use by sets of adjacent visits with common "note" values

"Note" values in the database indicate how the scheduler selected visits. Divide the survey into time intervals of consecutive visits with common notes, and assign color codes according to the filters and program indicated by the note. 

#### One month

The stock `UseMetric.run` in this version of `rubin_sim` assumes `data_slice[self.note_col]` is a scalar string, but `BlockIntervalSlicer` always passes an array of visits per block (even for a block with a single visit), so `.startswith()` fails with `AttributeError: 'numpy.ndarray' object has no attribute 'startswith'`. Patch `UseMetric.run` to use the first visit's note value in each block before calling `.startswith()`.

In [ ]:
# Patch UseMetric.run: BlockIntervalSlicer passes an array of visits per block,
# not a scalar, so data_slice[col] is always a numpy.ndarray (even length 1).
def _use_metric_run(self, data_slice, slice_point=None):
    prog_values = np.atleast_1d(data_slice[self.prog_col])
    note_values = np.atleast_1d(data_slice[self.note_col]).astype(str)

    if len(self.science_programs) > 0 and not np.isin(prog_values, self.science_programs).any():
        return "not science"

    note = note_values[0]
    for start_match in maf.metrics.UseMetric.start_matches:
        if note.startswith(start_match):
            return start_match

    return "other"


maf.metrics.UseMetric.run = _use_metric_run

In [ ]:
logger.info("Starting hourglass")
bundle_group = maf.metric_bundles.MetricBundleGroup(
    bundle_dict=[
        maf.metricBundles.MetricBundle(
            metric=maf.metrics.UseMetric(),
            slicer=maf.slicers.BlockIntervalSlicer(),
            constraint="",
            plot_dict={
                "title": f"Time usage for {MONTH}/{YEAR}",
                "legend_elements": (
                    "wide with u, g, or r",
                    "wide with only IR",
                    "greedy",
                    "ECDFS",
                    "EDFS",
                    "ELAISS1",
                    "COSMOS",
                    "COSMOS-transit",
                    "XMM-LSS",
                    "moon",
                ),
            },
            plot_funcs=[maf.plots.MonthHourglassUsePlot(MONTH, YEAR)],
        )
    ],
    db_con=baseline,
    out_dir=maf_output_dir,
    results_db=results_db,
)
bundle_group.run_all()
bundle_group.plot_all(closefigs=False)

###  Explication du crash :

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(baseline)

df = pd.read_sql(
    """
    SELECT science_program, scheduler_note, observation_reason, target_name
    FROM observations
    LIMIT 20
    """,
    conn,
)

print(df)

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(baseline)

df = pd.read_sql(
    """
    SELECT *
    FROM observations
    LIMIT 1
    """,
    conn,
)

print(df.columns.tolist())

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(baseline)

for col in ["observation_reason", "target_name", "science_program"]:
    print("\n======", col, "======")
    df = pd.read_sql(
        f"""
        SELECT {col}, COUNT(*) as n
        FROM observations
        GROUP BY {col}
        ORDER BY n DESC
        LIMIT 30
        """,
        conn,
    )
    print(df)

In [ ]:
def extract_fieldname(target_name):
    """ """

    target_name = str(target_name).lower()

    if "ddf_cosmos" in target_name:
        return "COSMOS"

    if "ddf_ecdfs" in target_name:
        return "ECDFS"

    if "ddf_edfs" in target_name:
        return "EDFS"

    if "ddf_elaiss1" in target_name:
        return "ELAIS-S1"

    if "ddf_xmm_lss" in target_name:
        return "XMM-LSS"

    return None


class Rubin53HourglassMetric(BaseMetric):
    def __init__(self, **kwargs):
        super().__init__(
            col=[
                "observation_reason",
                "target_name",
                "scheduler_note",
            ],
            metric_dtype="object",
            **kwargs,
        )

    def run(self, data_slice, slice_point=None):
        reasons = np.array(data_slice["observation_reason"], dtype=str)

        targets = np.array(data_slice["target_name"], dtype=str)

        # Twilight
        if np.any(reasons == "twilight_near_sun"):
            return "Twilight"

        # DDF avec champ
        for t in targets:
            field = extract_fieldname(t)
            if field is not None:
                return f"DDF-{field}"

        # ToO
        too = [
            "GW_case",
            "LensedBNS",
            "ToO",
        ]

        for p in too:
            if np.any(np.char.find(targets, p) >= 0):
                return "ToO"

        # templates
        if np.any(np.char.find(reasons, "template") >= 0):
            return "Template"

        # pairs
        if np.any(np.char.find(reasons, "pairs") >= 0):
            return "Pair"

        # singles
        if np.any(np.char.find(reasons, "singles") >= 0):
            return "Greedy"

        return "Other"

In [ ]:
metric = Rubin53HourglassMetric()

df = pd.read_sql(
    """
    SELECT scheduler_note,
           observation_reason,
           target_name
    FROM observations
    LIMIT 10000
    """,
    conn,
)

print(metric.run(df.to_records(index=False)))

In [ ]:
bundle_group = maf.metric_bundles.MetricBundleGroup(
    bundle_dict=[
        maf.metricBundles.MetricBundle(
            metric=metric,
            slicer=maf.slicers.BlockIntervalSlicer(),
            constraint="",
            plot_dict={
                "title": f"Time usage for {MONTH}/{YEAR}",
                "legend_elements": (
                    "DDF-COSMOS",
                    "DDF-ECDFS",
                    "DDF-EDFS",
                    "DDF-ELAIS-S1",
                    "DDF-XMM-LSS",
                    "Greedy",
                    "Other",
                    "Pair",
                    "Template",
                    "ToO",
                    "Twilight",
                ),
                "cmap": plt.get_cmap("tab20"),
                "assigned_colors": OrderedDict(
                    [
                        ("DDF-COSMOS", 0),
                        ("DDF-ECDFS", 1),
                        ("DDF-EDFS", 2),
                        ("DDF-ELAIS-S1", 3),
                        ("DDF-XMM-LSS", 4),
                        ("Greedy", 5),
                        ("Other", 6),
                        ("Pair", 7),
                        ("Template", 8),
                        ("ToO", 9),
                        ("Twilight", 10),
                    ]
                ),
                "legend_ncols": 1,
                "legend_loc": (1.01, 0.5),
                "legend_bbox_to_anchor": (1.01, 0.0),
                "legend": True,
            },
            # plot_funcs=[maf.plots.MonthHourglassUsePlot(MONTH, YEAR)],
            plot_funcs=[maf.plots.MonthHourglassCategoricalPlot(MONTH, YEAR)],
        )
    ],
    db_con=baseline,
    out_dir=maf_output_dir,
    results_db=results_db,
)

In [ ]:
bundle_group.run_all()
print(bundle_group.has_run)

In [ ]:
bundle = list(bundle_group.bundle_dict.values())[0]

print("type(bundle) = ", type(bundle))
print("bundle.metric.name = ", bundle.metric.name)

print("bundle.metric_values.compressed() = ", np.unique(bundle.metric_values.compressed()))

In [ ]:
bundle_group.plot_all(closefigs=False)

## Control what occur during the first year

In [ ]:
logger.info("Finished hourglass")

In [ ]:
bundles = []

for month in range(1, 13):
    bundle = maf.MetricBundle(
        metric=metric,
        slicer=maf.slicers.BlockIntervalSlicer(),
        constraint="",
        plot_funcs=[maf.plots.MonthHourglassCategoricalPlot(month, YEAR)],
        # run_name=run_name,
    )
    bundles.append(bundle)

bg = maf.MetricBundleGroup(
    {f"month_{m}": b for m, b in enumerate(bundles)},
    db_con=baseline,
    out_dir=maf_output_dir,
    results_db=results_db,
)

bg.run_all()
bg.plot_all()

In [ ]:
assert False

In this plot, the red represents scheduled "blobs" that include only exposures in IR (i, z, or y) filters, and the blue scheduled blobs with at least some exposures with filters covering shorter wavelengths. The orange shows times during which the "greedy" algorithm was used by the scheduler. Horizontal bars with other colors indicate times during which DDF fields were observed. The thick slanted yellow line shows the times of the transit of the moon, while the dotted yellow lines show moon rise and set. Slanted lines of other colors show  the transit times of each DDF field, with the same color coding as used for the bars that indicate when the DDFs are observed. The vertical axis indicates the day of the month on which the night starts, and the horizontal axis the time relative to local solar midnight (solar anti-transit). A white background indicates day time; black, full night; and different shades of gray, civil, nautical, and astronomical twilight.

If you wish to show the time relative to local civil midnight, you can pass the `solar_time=False` to the hourglass plotter:

In [ ]:
logger.info("Starting hourglass")
bundle_group = maf.metric_bundles.MetricBundleGroup(
    bundle_dict=[
        maf.metric_bundles.MetricBundle(
            metric=maf.metrics.UseMetric(),
            slicer=maf.slicers.BlockIntervalSlicer(),
            constraint="",
            plot_dict={
                "title": f"Time usage for {MONTH}, {YEAR}",
                "xMin": -6.5,
                "xMax": 8.5,
                "legend_elements": (
                    "wide with u, g, or r",
                    "wide with only IR",
                    "greedy",
                    "ECDFS",
                    "EDFS",
                    "COSMOS" "ELAISS1",
                    "XMM-LSS",
                    "moon",
                ),
            },
            plot_funcs=[maf.plots.MonthHourglassUsePlot(MONTH, YEAR, solar_time=False)],
        )
    ],
    db_con=baseline,
    out_dir=maf_output_dir,
    results_db=results_db,
)
bundle_group.run_all()
bundle_group.plot_all(closefigs=False)

Note the jump at the end of daylight savings time.

#### One Year

In [ ]:
%%time
logger.info("Starting hourglass")
bundle_group = maf.metric_bundles.MetricBundleGroup(
    bundle_dict=[
        maf.metric_bundles.MetricBundle(
            metric=maf.metrics.UseMetric(),
            slicer=maf.slicers.BlockIntervalSlicer(),
            constraint="",
            plot_dict={
                "figsize": (15, 10),
                "title": "Time usage for 2027",
                "legend_elements": (
                    "wide with u, g, or r",
                    "wide with only IR",
                    "greedy",
                    "ECDFS",
                    "EDFS",
                    "COSMOS" "ELAISS1",
                    "XMM-LSS",
                    "moon",
                ),
            },
            plot_funcs=[maf.plots.YearHourglassUsePlot(2027)],
        )
    ],
    db_con=baseline,
    out_dir=maf_output_dir,
    results_db=results_db,
)
bundle_group.run_all()
bundle_group.plot_all(closefigs=False)

#### Whole survey

Making plots for all years in the survey is slow, usually taking between 15 and 20 minutes on my test node.
This is done using separate instances of plotter objects, rather than just one, because the plot gets too compressed (or too big) otherwise.

In [ ]:
%%time
logger.info("Starting hourglass")
bundle_group = maf.metric_bundles.MetricBundleGroup(
    bundle_dict=[
        maf.metric_bundles.MetricBundle(
            metric=maf.metrics.UseMetric(),
            slicer=maf.slicers.BlockIntervalSlicer(),
            constraint="",
            plot_dict={
                "figsize": (15, 10),
                "legend_elements": (
                    "wide with u, g, or r",
                    "wide with only IR",
                    "greedy",
                    "ECDFS",
                    "EDFS",
                    "COSMOS" "ELAISS1",
                    "XMM-LSS",
                    "moon",
                ),
            },
            plot_funcs=[maf.plots.YearHourglassUsePlot(y) for y in range(2024, 2035)],
        )
    ],
    db_con=baseline,
    out_dir=maf_output_dir,
    results_db=results_db,
)
bundle_group.run_all()
bundle_group.plot_all(closefigs=False)

In [ ]:
logger.info("Finished hourglass")